# 第 13 章　综合项目实战

> **本章定位**：把前 12 章的文本处理能力升级为"画面"。第一次看到自己写的程序动起来，也是智能网联汽车 HMI 与交通仿真的入门形态。
>
> **知识点范围**：`turtle` 坐标系与画笔、矩形/圆点/文字绘制、填充、`tracer(0)` 与 `update()` 动画刷新；`pygame` 初始化、主循环、事件处理、帧率控制、`Rect` 碰撞检测与计分。
>
> **环境准备**：
> - `turtle` 是 Python 标准库，无需安装，但需要图形界面；
> - `pygame` 需安装：`pip install pygame`（国内可加镜像 `-i https://pypi.tuna.tsinghua.edu.cn/simple`）；
> - 在 Jupyter 中：`turtle` 会弹出独立窗口，绘图结束后点击窗口关闭；`pygame` 同理，需手动关闭或从菜单中断内核。
>
> **本章约定**：综合应用，允许使用此前所有知识点。

> **先修知识**：第 01—12 章全部语法；`turtle` 需要图形界面，`pygame` 需先执行 `pip install pygame`。无法弹出图形窗口的机器只做代码阅读与参数修改，不要求运行出画面。
>
> **学习目标**：完成本章后，你能够：
>
> 1. 用 `turtle` 画出含边框、填充与文字的交通场景，把截图与讲义示例逐项比对；
> 2. 用 `tracer(0)` 与 `update()` 实现动画刷新，说明不加 `tracer(0)` 时的绘制差异；
> 3. 搭出 `pygame` 主循环（初始化、事件处理、绘制、`clock.tick`），使窗口连续运行 10 秒不卡死；
> 4. 用 `Rect` 完成 1 次碰撞检测并按规则加分，打印碰撞前后的分数；
> 5. 修正 2 个常见故障（画面不刷新、退出事件未处理），说明每处修改对应的知识点。

---

## 本章学习路径

| 阶段 | 内容 | 编号 | 作用 |
|---|---|---|---|
| 第一部分 | 知识讲解 | 知识点 13.1 — 13.5 | 绘图坐标系、动画、碰撞 + 随堂示例 |
| 第二部分 | 代码示范 | 案例 13-1、案例 13-2 | 路口示意图 / 车流仿真 |
| 第三部分 | 布置练习 | 练习 13-1、练习 13-2 | 停车场平面图 / 碰撞计分 |
| 第四部分 | 习题答案 | 答案 13-1、答案 13-2 | 对照解析与评分要点 |

**知识点与代码对照表**：

| 知识点 | 名称 | 讲解位置 | 代码示范 | 对应练习 |
|---|---|---|---|---|
| 13.1 | turtle 坐标系与画笔 | 13.2 | 案例 13-1 | 练习 13-1 |
| 13.2 | 矩形、圆点、文字与填充 | 13.3 | 案例 13-1 | 练习 13-1 |
| 13.3 | 动画刷新：`tracer(0)` + `update()` | 13.4 | 案例 13-1 | 练习 13-1 |
| 13.4 | pygame 主循环与帧率控制 | 13.5 | 案例 13-2 | 练习 13-2 |
| 13.5 | 碰撞检测与计分 | 13.6 | 案例 13-2 | 练习 13-2 |

## 教学提示（教师用）

- **建议课时**：12 学时（6 次课），对应课程标准模块 13，建议安排为课程末的项目周。
- **讲解主线**：先用 `turtle` 建立"坐标—图形—刷新"的直觉（可视、可停顿、易调试），再进入 `pygame` 讲"主循环 + 帧率 + 碰撞"。**两个库的坐标系相反**（turtle 原点在中心、y 向上；pygame 原点在左上、y 向下），必须在开课时点明。
- **学生常见卡点**：turtle 忘记抬笔导致拖线；多个图形共用一个画笔时 `clear()` 互相擦除；pygame 每帧忘记 `fill()` 造成拖影；用固定 `sleep` 代替 `clock.tick` 导致不同机器速度不一致；窗口未关闭就反复运行，占满内核。
- **课堂活动建议**：分组调整 13-2 的红灯时长与车辆速度，观察停止线前排队长度变化，录屏汇报——把"参数—现象"的因果关系讲清楚，比讲语法更接近工程思维。

---

# 第一部分　知识讲解

## 13.1 场景导入：从"打印文字"到"画出画面"

前 12 章产出的是文本，本章产出的是画面：路口平面图、信号灯倒计时动画、车流仿真。绘图库把三件事抽象出来：

1. **坐标**：屏幕上的位置用 (x, y) 表示；
2. **颜色**：用色名或 `#RRGGBB` 十六进制表示；
3. **刷新**：画完之后告诉屏幕"现在重画一次"。

本章使用两个库，它们的坐标系**相反**，这是最容易出错的地方：

| 库 | 原点位置 | y 轴方向 | 适合 |
|---|---|---|---|
| `turtle` | 窗口**中心** | 向上为正 | 静态示意图、教学演示 |
| `pygame` | 窗口**左上角** | **向下**为正 | 实时仿真、交互 |

## 13.2 知识点 13.1　turtle 坐标系与画笔

### 13.2.1 核心概念

`turtle` 把画笔想象成一只海龟：它在坐标系中移动，移动轨迹就是线条。

| 操作 | 方法 | 说明 |
|---|---|---|
| 移动（不留痕） | `penup()` | 抬笔，之后移动不画线 |
| 移动（留痕） | `pendown()` | 落笔，之后移动画线 |
| 定位 | `goto(x, y)` | 直接跳到坐标点 |
| 方向移动 | `forward(d)` / `backward(d)` | 沿当前朝向前进/后退 |
| 转向 | `left(a)` / `right(a)` | 左转/右转 a 度 |
| 设置朝向 | `setheading(角度)` | 0 度朝右，90 度朝上 |
| 隐藏画笔 | `hideturtle()` | 只留图形，不留箭头 |
| 速度 | `speed(0)` | 0 为最快，绘图时不必看过程 |

**原理补充（概念深化）**：两套坐标系的冲突来自习惯：位图屏幕把左上角当原点、行号向下增大，是显存按行扫描的遗留；turtle 沿用数学坐标，原点在中心、纵轴向上。二者是固定的仿射变换：屏幕横坐标减去半宽得 turtle 横坐标，半高减去屏幕纵坐标得 turtle 纵坐标，纵轴项带负号正是「上下颠倒」的根因——屏幕上车往下开对应 turtle 纵坐标减小。换算只改参照系，画笔对象、朝向、已画图形都不变，坐标在内存里就是两个普通浮点数。常见误解是把定位当成搬运像素块，其实它只把笔尖设到新坐标，是否留线由抬笔落笔单独决定。

### 13.2.2 语法要点

- `turtle.Turtle()` 创建一个画笔对象，**可创建多个**，各自独立（案例 13-1 用了三个：场景、灯、文字）；
- `turtle.Screen()` 是窗口对象，负责标题、尺寸、背景色；
- 角度以**逆时针**为正，0 度指向右（+x）；
- 所有坐标都是**画笔中心**的位置，`goto` 到某点即笔尖到达该点。

### 13.2.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 忘记 `penup()` | 移动时拖出一条线 | 定位前先抬笔 |
| 多个图形共用一个画笔 | 后画的覆盖先画的 | 分画笔，或按顺序画 |
| 坐标系按屏幕像素理解 | 图形位置与预期上下颠倒 | 记住中心为原点、y 向上 |
| 绘制后窗口立刻关闭 | 看不到结果 | 末尾 `screen.exitonclick()` |

### 13.2.4 随堂示例：屏幕坐标 ↔ turtle 坐标换算

**核心要点**：验证两套坐标系的差别：屏幕以左上角为原点、y 向下；turtle 以窗口中心为原点、y 向上。示例用纯计算演示换算，不弹窗即可看到结果。

In [ ]:
W, H = 640, 480

def to_turtle(px, py):
    """把"左上角原点、y 向下"的像素坐标换算为 turtle 坐标"""
    return px - W / 2, H / 2 - py

print("屏幕中心 (320, 240) →", to_turtle(320, 240))
print("屏幕左上 (0, 0)     →", to_turtle(0, 0))
print("屏幕右下 (640, 480) →", to_turtle(640, 480))
print("道路中线 y=240 的 turtle y：", to_turtle(0, 240)[1])

**预期输出**：

```text
屏幕中心 (320, 240) → (0.0, 0.0)
屏幕左上 (0, 0)     → (-320.0, 240.0)
屏幕右下 (640, 480) → (320.0, -240.0)
道路中线 y=240 的 turtle y： 0.0
```

**变式代码（迁移练习）**：本段用纯计算演示原理，不弹图形窗口：把随堂示例的「屏幕→turtle」单向换算迁移为双向换算并代入车辆下行轨迹，验证 y 轴翻转下「屏幕 y 增大 = turtle y 减小」以及往返可还原。

In [ ]:
W, H = 800, 600  # 路口画布尺寸（像素）
def to_turtle(px, py): return px - W / 2, H / 2 - py  # 屏幕→turtle：y 轴翻转
def to_screen(tx, ty): return tx + W / 2, H / 2 - ty  # 反向换算
road = [(400, 100), (400, 200), (400, 300)]  # 车辆沿屏幕 y 增大方向下行
turtle_pts = [to_turtle(px, py) for px, py in road]
print("turtle 坐标（y 随下行变小）：", turtle_pts)
print("换算回屏幕：", [to_screen(tx, ty) for tx, ty in turtle_pts])

**预期输出**：

```text
turtle 坐标（y 随下行变小）： [(0.0, 200.0), (0.0, 100.0), (0.0, 0.0)]
换算回屏幕： [(400.0, 100.0), (400.0, 200.0), (400.0, 300.0)]
```

### 13.2.5 实操步骤（上机跟随）

1. 运行观察 `turtle 坐标` 依次为 `[(0.0, 200.0), (0.0, 100.0), (0.0, 0.0)]`，y 随车辆下行而变小。
2. 核对 x 恒为 0.0：采样点屏幕 x=400 恰是 W/2，换算后落在 turtle 的 y 轴上。
3. 观察 `换算回屏幕` 得到 `[(400.0, 100.0), (400.0, 200.0), (400.0, 300.0)]`，数值与原 `road` 三点一致。
4. 把 `H` 由 600 改成 480 再运行，观察 turtle y 整体偏移 60，理解 `H/2` 是原点平移量。
5. 把 `H / 2 - py` 误写成 `py - H / 2`，观察 y 正负翻转，复现「图形上下颠倒」。
6. 把采样点加上 `(0, 0)`，观察换算结果为 `(-400.0, 300.0)`，验证左上角对应 turtle 的左下方向。

### 13.2.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| 画出来的路口上下颠倒 | 按屏幕「y 向下为正」理解了 turtle 坐标 | 记住 turtle 原点在中心、y 向上，用 `H/2 - py` 换算 |
| `goto` 之前拖出一条多余的线 | 定位前忘记 `penup()` | 先抬笔再 `goto`，落笔后才画线 |
| 多个图形互相覆盖或被擦掉 | 多个元素共用一支画笔，`clear()` 连带擦除 | 场景、灯、文字各用一支 `turtle.Turtle()` |
| 窗口一闪就关闭看不到结果 | 绘图结束没有保持窗口 | 末尾加 `screen.exitonclick()` |
| 换算后坐标差了半屏 | 忘了减 `W/2`、`H/2` 把原点平移到中心 | 正向 `px - W/2, H/2 - py`，反向 `tx + W/2, H/2 - ty` |

## 13.3 知识点 13.2　矩形、圆点、文字与填充

### 13.3.1 核心概念

| 图形 | 方法 | 要点 |
|---|---|---|
| 矩形 | `forward` + `left(90)` 循环两次 | 必须成对，共 4 条边 |
| 圆点 | `dot(直径, 颜色)` | 适合表示信号灯、车位标记 |
| 文字 | `write(内容, align=, font=)` | 需指定字体与字号 |
| 填充 | `begin_fill()` … `end_fill()` | 两者之间的闭合路径被填充 |

**封装的必要性**：画一个矩形要 8 行（抬笔、定位、落笔、设色、开始填充、两次"前进+左转"、结束填充），重复写十几次既冗长又易错。案例 13-1 把它封装成 `rect(t, x, y, w, h, color)`，用 6 个参数描述一个矩形。

**原理补充（概念深化）**：填充要求闭合路径，是因为开始填充与结束填充之间记录的是一串顶点，库用扫描线算法给这串点围成的多边形上色；只给四个角点时，库会自动把末点连回起点补上第五条边，所以结束填充隐含一次「回到起点」。顶点须按统一顺时针排列，顺序错乱会让多边形自交、填充溢出。中心等于两条对角顶点的平均值，因为矩形是平行四边形、对角线互相平分。把顶点翻译成画笔指令就是「前进长、左转九十度、前进宽、左转九十度」循环两遍，四次左转累计三百六十度正好回到初始朝向，这正是封装矩形函数里循环两次的来历。

### 13.3.2 语法要点

| 要素 | 写法 | 说明 |
|---|---|---|
| 边框色 | `t.color("#5b5b5b")` | 同时影响后续线条与文字 |
| 填充色 | `t.fillcolor(color)` | 与边框色可不同 |
| 填充范围 | `begin_fill()` … `end_fill()` | 顺序不能颠倒，否则不填充 |
| 字体 | `font=("Microsoft YaHei", 14, "bold")` | 中文用雅黑，英文可用 Arial |
| 对齐 | `align="center"` | 以当前坐标为文字中心 |

### 13.3.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| `end_fill()` 写在 `begin_fill()` 前 | 不填充 | 顺序改正 |
| 填充路径未闭合 | 填充溢出到别处 | 保证回到起点 |
| 字体名不存在 | 中文显示为方块 | 用系统已装字体（雅黑/黑体） |
| 文字位置按左上角算 | 文字偏移 | `align` 决定对齐基准 |

### 13.3.4 随堂示例：矩形的四个顶点与填充路径

**核心要点**：验证 `rect()` 的填充路径怎么给：4 个顶点按顺时针排列，坐标规律是「x 加宽、y 加高」，中心点等于两组对角的平均值；列出的 4 个点尚未回到起点，最后那条边由 `end_fill()` 自动闭合。

In [ ]:
def rect_corners(x, y, w, h):
    """以 (x, y) 为左下角，按顺时针返回填充路径的 4 个顶点"""
    return [(x, y), (x + w, y), (x + w, y + h), (x, y + h)]

corners = rect_corners(-100, -50, 200, 100)
for i, p in enumerate(corners, start=1):
    print("顶点", i, "：", p)

# 4 个点只走完一圈，末点还没回到起点：最后那条边由 end_fill() 自动闭合
print("末点与起点重合吗：", corners[-1] == corners[0])
print("缺的那条边：从", corners[-1], "回到", corners[0])
print("中心坐标：", ((corners[0][0] + corners[2][0]) / 2,
                    (corners[0][1] + corners[2][1]) / 2))

**预期输出**：

```text
顶点 1 ： (-100, -50)
顶点 2 ： (100, -50)
顶点 3 ： (100, 50)
顶点 4 ： (-100, 50)
末点与起点重合吗： False
缺的那条边：从 (-100, 50) 回到 (-100, -50)
中心坐标： (0.0, 0.0)
```

**变式代码（迁移练习）**：本段用纯计算演示原理，不弹图形窗口：把随堂示例的「列出 4 个顶点」迁移为「顶点 → 画笔指令序列 + 中心 + 周长 + 自动闭合边」，还原真实 `rect()` 里 `forward + left(90)` 循环两次的几何依据。

In [ ]:
def rect_path(x, y, w, h):  # 由左下角生成顺时针顶点、画笔指令与中心
    corners = [(x, y), (x + w, y), (x + w, y + h), (x, y + h)]
    steps = [(w, 90), (h, 90), (w, 90), (h, 90)]  # 前进长度+左转角度，共 4 边
    center = ((corners[0][0] + corners[2][0]) / 2, (corners[0][1] + corners[2][1]) / 2)
    return corners, steps, center
c, s, mid = rect_path(-60, -30, 120, 60)  # 路口中心区矩形
print("顺时针 4 顶点：", c)
print("画笔指令(前进,左转)：", s)
print("闭合边：", c[-1], "→", c[0], "由 end_fill() 自动补齐")
print("中心：", mid, "；周长：", sum([d for d, a in s]))

**预期输出**：

```text
顺时针 4 顶点： [(-60, -30), (60, -30), (60, 30), (-60, 30)]
画笔指令(前进,左转)： [(120, 90), (60, 90), (120, 90), (60, 90)]
闭合边： (-60, 30) → (-60, -30) 由 end_fill() 自动补齐
中心： (0.0, 0.0) ；周长： 360
```

### 13.3.5 实操步骤（上机跟随）

1. 运行观察 `顺时针 4 顶点` 为 `[(-60,-30), (60,-30), (60,30), (-60,30)]`，x 先增大后不变、y 先不变后增大。
2. 观察 `画笔指令` 为 `[(120, 90), (60, 90), (120, 90), (60, 90)]`，即长宽交替、每次左转 90°。
3. 核对 4 个左转累计 360°，说明画完回到初始朝向，路径闭合。
4. 观察 `闭合边： (-60, 30) → (-60, -30)`，确认 4 个点未回到起点、末边由 `end_fill()` 补齐。
5. 观察 `中心： (0.0, 0.0)` 与 `周长： 360`，核对中心是 p0 与 p2 的平均值、周长是 `2×(120+60)`。
6. 把 `w` 改成 200 再运行，观察第 2、3 个顶点 x 变 140、周长变 520，验证参数只影响尺寸。

### 13.3.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| 矩形没有填充颜色 | `end_fill()` 写在 `begin_fill()` 之前，顺序颠倒 | 先 `begin_fill()` 画路径，再 `end_fill()` |
| 填充溢出到别的图形上 | 填充路径未回到起点、未闭合 | 保证 4 条边闭合，让 `end_fill()` 补齐末边 |
| 中文显示成方块 | 字体名系统里不存在 | 用已安装字体如 `Microsoft YaHei` / `SimHei` |
| 文字位置整体偏移 | 按左上角理解 `write` 的定位基准 | 用 `align` 设为 `center`，以当前坐标为文字中心 |
| 十几个矩形代码又长又易错 | 每个图形都重写抬笔、定位、填充 | 封装 `rect(t, x, y, w, h, color)` 一次调用 |

## 13.4 知识点 13.3　动画刷新

### 13.4.1 核心概念

turtle 默认"每画一笔就刷新一次"，图形多时会闪烁、变慢。标准做法：

```text
screen.tracer(0)     # 关闭自动刷新
... 绘制大量图形 ...
screen.update()      # 手动一次性刷新
```

做动画时，每帧"清空 → 重画 → update"：`turtle.Turtle().clear()` 只清除**该画笔**画的内容，因此"场景 / 灯 / 文字"要分属不同画笔，才能只更新需要变化的部分。

**原理补充（概念深化）**：turtle 默认每画一笔就刷新一次，几十个矩形要刷几十次，人眼会看到逐笔蹦出并闪烁。标准做法是双缓冲：先关闭自动刷新，把整帧绘制写进离屏缓冲，帧末一次手动刷新贴到屏幕，于是多笔绘制只对应一次刷新——次数从「笔画数」降到「帧数」，这就是八秒每帧八笔算出的六十四次变八次。每帧要遵循「清空、重画、手动刷新」三步：清除只擦该画笔的内容，故场景、灯、文字须分属不同画笔。忘记清除时新旧数字会叠加成 321。缩短暂停不能提速，卡顿来自重绘面积与刷新次数，正解是减少重绘量。

### 13.4.2 语法要点

| 需求 | 写法 |
|---|---|
| 关闭自动刷新 | `screen.tracer(0)` |
| 手动刷新 | `screen.update()` |
| 暂停指定秒数 | `time.sleep(1)`（需 `import time`） |
| 清空某画笔 | `t.clear()`（不影响其他画笔） |
| 倒计时循环 | `for s in range(3, 0, -1):` 从 3 到 1 |

### 13.4.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 关了 `tracer` 忘记 `update` | 窗口一片空白 | 每帧结束 `update()` |
| 只用一支笔画所有元素 | `clear()` 把场景也擦掉 | 分画笔 |
| 用 `sleep(0.01)` 试图提速 | 动画仍卡顿 | 减少重绘面积，而非缩短间隔 |
| 循环内不清屏 | 数字叠在一起 | 每帧先 `clear()` 再 `write()` |

### 13.4.4 随堂示例：tracer(0) + update() 的刷新节奏

**核心要点**：验证动画原理：关闭自动刷新后，每次改完内容只调用一次 `update()`，一秒一帧；若忘记 `clear()`，上一帧的数字会与新的叠加。

In [ ]:
# 对应 screen.tracer(0)：先关掉自动刷新
frame = 0
for s in range(3, 0, -1):
    frame += 1
    # 对应 text.clear() + text.write(str(s))
    print("第", frame, "帧：重画数字", s, "→ 调用 update() 刷新")
print("倒计时结束：重画提示「绿灯放行」→ 调用 update()")

# 若不清屏，数字会叠加：这里用字符串模拟"叠加"效果
overlay = ""
for s in range(3, 0, -1):
    overlay += str(s)
print("忘记 clear() 的结果：", overlay)

**预期输出**：

```text
第 1 帧：重画数字 3 → 调用 update() 刷新
第 2 帧：重画数字 2 → 调用 update() 刷新
第 3 帧：重画数字 1 → 调用 update() 刷新
倒计时结束：重画提示「绿灯放行」→ 调用 update()
忘记 clear() 的结果： 321
```

**变式代码（迁移练习）**：本段用纯计算演示原理，不弹图形窗口：把随堂示例的逐帧打印迁移为「帧数 / 自动刷新次数 / tracer(0) 刷新次数」的量化对照，并复现忘记 `clear()` 时的数字叠加。

In [ ]:
FPS, strokes = 1, 8  # 每秒 1 帧；每帧约 8 笔绘制
red, green = 3, 5
frames = (red + green) * FPS  # 8 秒 → 8 帧
print("倒计时序列：", list(range(red, 0, -1)), "+", list(range(green, 0, -1)))
print("总帧数：", frames, "；自动刷新：", frames * strokes, "次；tracer(0)：", frames, "次")
# 每帧先 clear() 再 write()：忘了 clear 数字会叠加
overlay = ""
for s in range(red, 0, -1): overlay += str(s)
print("忘记 clear() 叠加：", overlay)

**预期输出**：

```text
倒计时序列： [3, 2, 1] + [5, 4, 3, 2, 1]
总帧数： 8 ；自动刷新： 64 次；tracer(0)： 8 次
忘记 clear() 叠加： 321
```

### 13.4.5 实操步骤（上机跟随）

1. 运行观察 `倒计时序列` 为 `[3, 2, 1] + [5, 4, 3, 2, 1]`，对应红灯 3 秒、绿灯 5 秒共 8 帧。
2. 观察 `总帧数： 8` 与 `自动刷新： 64`，核对 64 = 8 帧 × 每帧 8 笔。
3. 观察 `tracer(0)： 8`，确认手动刷新把次数从 64 降到 8（每帧仅 1 次 `update()`）。
4. 观察 `忘记 clear() 叠加： 321`，理解每帧不擦旧数字会逐帧拼接。
5. 把 `strokes` 由 8 改成 20 再运行，观察 `自动刷新` 变 160 而 `tracer(0)` 仍是 8，验证刷新次数只与帧数有关。
6. 把 `green` 改成 3 再运行，观察总帧数变 6、`tracer(0)` 同步变 6，确认帧数决定刷新节奏。

### 13.4.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| 窗口一片空白 | 开了 `tracer(0)` 却忘了 `update()` | 每帧末尾调用一次 `screen.update()` |
| `clear()` 把静态场景也擦掉了 | 所有元素共用一支画笔 | 场景、灯、文字分属不同 `Turtle()` |
| 倒计时数字叠成一团 | 每帧没先 `clear()` 再 `write()` | 固定「清空 → 重画 → update」三步 |
| 缩短 `sleep` 动画依旧卡顿 | 卡顿来自重绘面积与刷新次数，不是间隔 | 减少每帧重绘量并用 `tracer(0)` 合并刷新 |
| 关了自动刷新后图形仍逐笔出现 | `tracer(0)` 未生效或被在循环里反复重置 | 在 `Screen()` 上设置一次即可 |

## 13.5 知识点 13.4　pygame 主循环与帧率控制

### 13.5.1 核心概念

pygame 程序的骨架固定为六步：

```text
初始化 → 建窗口 → 主循环（事件 → 更新 → 绘制 → flip）
```

| 步骤 | 代码 | 作用 |
|---|---|---|
| 初始化 | `pygame.init()` | 初始化各子系统 |
| 建窗口 | `pygame.display.set_mode((W, H))` | 设定画布尺寸 |
| 帧率 | `clock = pygame.time.Clock()` + `clock.tick(60)` | 限制每秒帧数，返回距上一帧毫秒数 |
| 事件 | `for event in pygame.event.get():` | 处理关闭、按键 |
| 绘制 | `pygame.draw.rect/line/circle` | 画图元 |
| 刷新 | `pygame.display.flip()` | 把画好的内容显示出来 |

**时间驱动而非帧驱动**：`dt = clock.tick(FPS) / 1000.0` 得到秒数，位移写 `x += speed * 60 * dt`，这样在不同性能的机器上速度一致。若直接写 `x += speed`，帧率高的机器上车会跑得更快。

**原理补充（概念深化）**：六步骨架里最易错的是「更新」：若写每帧固定步长（帧驱动），同一程序在三十、六十、一百二十帧率下分别跑九十、一百八十、三百六十像素，帧率越高车越快，结果随硬件漂移。正解是把位移建立在真实时间上：`clock.tick(FPS)` 返回距上帧毫秒数，换算成秒得到 `dt`，位移写「速度乘 dt」；帧率翻倍则帧数翻倍、每帧 dt 减半，累加恒等于「速度乘总秒数」，与帧率无关，这就是解耦。相位切换同理用累计 `timer += dt` 而非帧计数，才能保证「绿灯五秒」在快慢机器上都是五秒。

### 13.5.2 语法要点

| 图形 | 函数 |
|---|---|
| 矩形 | `pygame.draw.rect(screen, 颜色, (x, y, w, h), border_radius=6)` |
| 直线 | `pygame.draw.line(screen, 颜色, (x1, y1), (x2, y2), 线宽)` |
| 圆 | `pygame.draw.circle(screen, 颜色, (cx, cy), 半径)` |
| 文字 | `screen.blit(font.render(文本, True, 颜色), (x, y))` |

### 13.5.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 循环内不处理事件 | 窗口无响应（"转菊花"） | 每帧 `pygame.event.get()` |
| 忘记 `flip()` | 窗口一片黑 | 每帧末尾刷新 |
| 忘记 `screen.fill()` | 上一帧残留拖影 | 每帧先清屏 |
| 用帧数控制速度 | 不同机器速度不同 | 用 `dt` |
| 忘记 `pygame.quit()` | 退出时资源未释放 | 循环后调用 |

### 13.5.4 随堂示例：用 dt 让位移与帧率解耦

**核心要点**：验证时间驱动位移：每帧位移 = 速度 × `dt`，而 `dt` 来自 `clock.tick(FPS)`；这样换机器、掉帧都不会改变实际行驶速度。示例用固定 `dt` 演算，不打开窗口。

In [ ]:
FPS = 60
GREEN_T = 0.05                 # 为便于演示，把绿灯压缩到 3 帧
dt = 1 / FPS                   # 每帧时长（秒）

speed = 3.0                    # 像素/帧
timer = 0.0
state = "green"
x = 0.0

for frame in range(1, 7):
    timer += dt
    if timer >= GREEN_T:
        timer = 0
        state = "yellow" if state == "green" else "green"
    x += speed * 60 * dt       # 位移 = 速度 × 60 × dt
    print("第", frame, "帧  相位", state, "  x =", round(x, 2))

**预期输出**：

```text
第 1 帧  相位 green   x = 3.0
第 2 帧  相位 green   x = 6.0
第 3 帧  相位 yellow   x = 9.0
第 4 帧  相位 yellow   x = 12.0
第 5 帧  相位 yellow   x = 15.0
第 6 帧  相位 green   x = 18.0
```

**变式代码（迁移练习）**：本段用纯计算演示原理，不弹图形窗口：把随堂示例固定 `dt` 的单机演算迁移为「同一段 1 秒路程在 30/60/120 FPS 下对比」，验证 dt 驱动的位移恒为 180 像素、帧驱动的位移随帧率漂移。

In [ ]:
def drive(fps, seconds, px_per_frame, by_dt):  # 同一段路程的两种写法
    x, dt = 0.0, 1 / fps
    for _ in range(int(fps * seconds)):
        x += px_per_frame * 60 * dt if by_dt else px_per_frame  # dt 驱动 vs 帧驱动
    return round(x, 2)
for fps in (30, 60, 120):
    print("FPS", fps, "：dt 驱动 1 秒 =", drive(fps, 1, 3, True),
          "px；帧驱动 =", drive(fps, 1, 3, False), "px")

**预期输出**：

```text
FPS 30 ：dt 驱动 1 秒 = 180.0 px；帧驱动 = 90.0 px
FPS 60 ：dt 驱动 1 秒 = 180.0 px；帧驱动 = 180.0 px
FPS 120 ：dt 驱动 1 秒 = 180.0 px；帧驱动 = 360.0 px
```

### 13.5.5 实操步骤（上机跟随）

1. 运行观察三行 `dt 驱动 1 秒` 全部是 `180.0` px，说明与 FPS 无关。
2. 观察 `帧驱动` 依次为 90.0、180.0、360.0 px，确认帧率翻倍位移也翻倍。
3. 核对 `3 * 60 * dt` 中 `3*60 = 180` 是像素/秒，累加 `fps` 个 `1/fps` 秒正好 1 秒。
4. 把 `seconds` 由 1 改成 2 再运行，观察 dt 驱动位移变 360.0，帧驱动变 180.0/360.0/720.0。
5. 把 60 FPS 的帧驱动结果 180.0 与 dt 驱动 180.0 对比，说明 60 FPS 恰是「名义帧率」的基准点。
6. 把 `px_per_frame * 60 * dt` 改成 `px_per_frame * dt` 再运行，观察位移缩成 3.0，理解「×60」是把 像素/帧 折算成 像素/秒。

### 13.5.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| 不同机器上车速明显不一样 | 用 `x += speed` 帧驱动，速度随帧率变化 | 用 `dt = clock.tick(FPS)/1000.0`，写 `x += speed*60*dt` |
| 窗口无响应 / 转菊花 | 主循环里没有 `pygame.event.get()` 处理事件 | 每帧取事件并响应 `QUIT` |
| 窗口一片黑 | 忘记 `pygame.display.flip()` | 每帧绘制末尾提交画面 |
| 上一帧残影拖在画面里 | 每帧开头没有 `screen.fill()` | 每帧先清屏再绘制 |
| 绿灯「5 秒」时快时慢 | 用帧计数而非累计 `dt` 控制相位 | 用 `timer += dt`，按累计秒数切换相位 |

## 13.6 知识点 13.5　碰撞检测与计分

### 13.6.1 核心概念

pygame 用 `Rect` 表示矩形区域，`Rect.colliderect(另一个Rect)` 返回两个矩形是否重叠。

车流仿真中的"追尾"判定要点：

1. **只比较同一车道**的相邻两车——跨车道比较会把对向车辆误判为追尾；
2. 先把同车道车辆**按 x 排序**，再比较相邻两辆；
3. 碰撞后把后车速度降为 0 并回退到安全距离，避免持续重叠导致每帧都计一次碰撞。

**原理补充（概念深化）**：矩形重叠判定的本质不是两点相碰，而是横、纵两组区间同时重叠：横向要「甲的左边界小于乙的右边界且乙的左边界小于甲的右边界」，纵向同理，四个条件全真才碰撞，有一轴分离就不算，用中心点距离会把长车误判。为何必须「按车道分组、按横坐标排序、只比相邻两车」？跨车道比较会把对向车算成追尾，不排序则「相邻」是列表顺序而非前后位置；排序后真正可能追尾的只可能是相邻两辆。碰撞后还须回退位置：只把速度置零，矩形仍重叠，下一帧判定依旧为真，碰撞数会每帧加一（六十帧率下一秒六十次）。

### 13.6.2 语法要点

| 操作 | 写法 |
|---|---|
| 构造矩形 | `pygame.Rect(x, y, w, h)` |
| 重叠判断 | `r1.colliderect(r2)` → `True` / `False` |
| 按车道分组 | 字典 `{y: [车辆…]}`，或列表推导式筛选 |
| 按位置排序 | `same.sort(key=lambda c: c["x"])`（第 7 章） |

### 13.6.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 不分车道比较 | 对向车被判追尾，碰撞数暴涨 | 按 `y` 分组 |
| 不排序就比较 | 顺序错乱，漏判 | 先按 `x` 排序 |
| 碰撞后不处理速度 | 每帧重复计碰撞 | 后车减速并回退 |
| 用中心点距离判重叠 | 长车误判 | 用矩形 `colliderect` |

---

### 13.6.4 随堂示例：矩形重叠判定与计分

**核心要点**：验证碰撞检测的本质是两组区间同时重叠，等价于 `pygame.Rect.colliderect()`；碰撞后扣分、未碰撞不加分，与案例的计分逻辑一致（示例不依赖 pygame）。

In [ ]:
def overlap(a, b):
    """两个矩形是否重叠：a、b 均为 (x, y, w, h)，等价于 colliderect()"""
    ax, ay, aw, ah = a
    bx, by, bw, bh = b
    return ax < bx + bw and bx < ax + aw and ay < by + bh and by < ay + ah

car1 = (100, 175, 40, 30)      # 前车
car2 = (120, 175, 40, 30)      # 后车：与 car1 重叠 → 追尾
car3 = (200, 175, 40, 30)      # 第三辆：与前车有间距

score = 0
if overlap(car1, car2):
    print("car1 与 car2 重叠 → 判定追尾，扣 5 分")
    score -= 5
else:
    print("car1 与 car2 未重叠 → 正常通行")

if overlap(car1, car3):
    print("car1 与 car3 重叠 → 扣 5 分")
    score -= 5
else:
    print("car1 与 car3 未重叠 → 得分不变")

print("最终得分：", score)

**预期输出**：

```text
car1 与 car2 重叠 → 判定追尾，扣 5 分
car1 与 car3 未重叠 → 得分不变
最终得分： -5
```

**变式代码（迁移练习）**：本段用纯计算演示原理，不弹图形窗口：把随堂示例的「三辆车两两判定」迁移为「按车道分组 → 按 x 排序 → 只比相邻两车 + 计分」，验证跨车道 x 相近不误判、同车道重叠才扣分。

In [ ]:
def hit(a, b):  # 两轴区间都重叠才判追尾，等价于 Rect.colliderect()
    return (a[0] < b[0]+b[2] and b[0] < a[0]+a[2] and a[1] < b[1]+b[3] and b[1] < a[1]+a[3])
# 车辆 = (x, y, w, h, 车道)，x 放首位便于直接 sorted
cars = [(100, 175, 40, 30, 1), (120, 175, 40, 30, 1), (105, 225, 40, 30, 2)]
score = 0
for lane in (1, 2):  # 按车道分组，再按 x 排序，只比较相邻两车
    same = sorted([c for c in cars if c[4] == lane])
    for i in range(len(same) - 1):
        if hit(same[i], same[i+1]): score -= 5; print("车道", lane, "追尾，扣 5")
print("跨车道 x 相近但不比较；最终得分：", score)

**预期输出**：

```text
车道 1 追尾，扣 5
跨车道 x 相近但不比较；最终得分： -5
```

### 13.6.5 实操步骤（上机跟随）

1. 运行观察只打印一行 `车道 1 追尾，扣 5`，说明只有车道 1 的相邻两车被判定。
2. 核对车道 2 的 `(105, 225, ...)` 与车道 1 首车 x=100 极近却未判追尾，确认按 `lane` 分组生效。
3. 观察 `最终得分： -5`，核对 `hit()` 的四个区间条件在 (100,175,40,30) 与 (120,175,40,30) 上全为真。
4. 观察 `sorted` 后车道 1 的顺序是 100 → 120，确认「相邻」是位置上的前后车而非列表原序。
5. 把第二辆车 x 由 120 改成 200 再运行，观察不再打印追尾、得分变 0，确认区间分离即不碰撞。

### 13.6.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| 碰撞数每秒涨几十 | 碰撞后只减速没回退位置，下一帧仍重叠 | 后车 x 退到前车之后（如 `前车x - 42`），再置 v=0 |
| 对向车道也被判追尾、碰撞数暴涨 | 未按车道 y 分组就全局两两比较 | 先按 `y` 分组，只比较同车道 |
| 排序后仍漏判或顺序错乱 | 没按 x 排序，「相邻」是列表顺序不是位置顺序 | 用 `sorted` 按 x 排序后再比相邻 |
| 长车擦肩而过被误判 | 用中心点距离判重叠 | 用矩形区间 `colliderect`，要求两轴同时重叠 |
| 注释写了「扣 5 分」但得分没变 | 碰撞分支里漏写 `score -= 5` | 把扣分语句写进 `if` 分支内 |

# 第二部分　代码示范

## 案例 13-1　turtle 绘制智慧路口与信号灯倒计时

- **覆盖知识点**：13.1（坐标系、画笔）、13.2（矩形、圆点、文字、填充）、13.3（`tracer` / `update` / 分画笔）
- **源文件**：`case1_路口示意图.py`
- **难度**：综合　**建议用时**：40 分钟

### 任务情境

为新员工培训制作一张路口平面示意图（含道路、中心区、四处斑马线、信号灯），并让信号灯完成一次"红灯 3 秒 → 绿灯 5 秒"的倒计时动画。

### 任务要求

1. 用 `rect()` 封装矩形绘制，统一处理"抬笔—定位—填充—落笔"；
2. 坐标系采用屏幕中心为原点的数学坐标系（turtle 默认）；
3. 用 `tracer(0)` 关闭自动刷新，在关键帧调用 `update()`，避免闪烁；
4. 用独立的海龟对象分别绘制场景、信号灯、文字，避免互相清除。

> ⚠️ **运行提示**：本段使用 `turtle`，运行后会弹出独立窗口。倒计时结束后点击窗口任意位置关闭。

### 输入规格

| 项 | 要求 |
|---|---|
| 数据来源 | 无外部输入；尺寸、颜色、标题与倒计时秒数均为源码字面量 |
| 类型 | `int` 与 `str` 字面量；`rect()` 的坐标与尺寸为 `int` 或 `float`；颜色为 `#RRGGBB` 或色名 |
| 取值范围 | `W` 为 640、`H` 为 480，窗口 `setup(680, 520)`；红灯 3 秒、绿灯 5 秒；斑马线每侧 6 条，条纹 9×90，间距 16 像素 |
| 非法值 | 用 `input()` 读秒数；`range(6)` 改成 `range(0)`；`range(seconds, 0, -1)` 写成 `range(seconds)` |

### 输出规格

| 项 | 要求 |
|---|---|
| 行数 | 终端恰好 1 行；其余内容全部画在 turtle 窗口，不进入终端 |
| 逐行格式 | 唯一一行为 `示意图绘制完成，点击窗口关闭`，行首尾无多余字符 |
| 窗口尺寸 | `screen.setup(680, 520)`，即 640×480 画布外扩 40 像素；标题 `智慧路口示意图 - turtle` |
| 绘制元素 | 横向灰色主干道 640×100、纵向灰色支路 100×480、棕色中心区 100×100、四处斑马线共 24 条、左上标题、路口上方 (0, 105) 直径 30 圆点、(0, 150) 处数字与提示语 |
| 刷新次数 | `tracer(0)` 关自动刷新；红灯 4 次、绿灯 6 次、收尾 1 次，共 11 次；`draw_lamp()` 只 `clear()` |
| 精度 / 单位 | 坐标与尺寸单位为像素，横向 ±320、纵向 ±240；倒计时为整秒 |
| 关闭方式 | 末尾执行 `screen.exitonclick()`，点击窗口任意位置关闭 |
| 判定方式 | 终端无报错，1 行逐字比对；窗口按上述尺寸与元素截图对照；`update()` 共 11 次 |

### 参考代码

In [ ]:
# ---------- 第 13 章 综合项目实战 · 案例 13-1：turtle 绘制智慧路口与信号灯倒计时 ----------
# 对应知识点：13.1 turtle 坐标系与画笔、13.2 矩形 / 圆点 / 文字 / 填充、13.3 tracer(0) + update() 动画刷新
# 功能说明：绘制路口平面图（道路、中心区、斑马线），并播放「红灯 3 秒 → 绿灯 5 秒」倒计时动画

import turtle
import time

# 画布尺寸：turtle 以窗口中心为原点，故所有坐标都落在 ±W/2、±H/2 之内
W, H = 640, 480


def rect(t, x, y, w, h, color):
    """以 (x, y) 为左下角画填充矩形"""
    t.penup()
    t.goto(x, y)
    t.pendown()
    t.color(color)
    t.fillcolor(color)
    t.begin_fill()
    for _ in range(2):
        t.forward(w)
        t.left(90)
        t.forward(h)
        t.left(90)
    t.end_fill()


def draw_scene(t):
    """绘制路口：两条道路 + 中心区 + 四处斑马线"""
    rect(t, -W / 2, -50, W, 100, "#5b5b5b")      # 横向主干道
    rect(t, -50, -H / 2, 100, H, "#5b5b5b")      # 纵向支路
    rect(t, -50, -50, 100, 100, "#8a6d0b")       # 路口中心区

    # 左右两侧斑马线（竖条纹）
    for i in range(6):
        rect(t, 62 + i * 16, -45, 9, 90, "white")
        rect(t, -62 - i * 16 - 9, -45, 9, 90, "white")

    # 上下两侧斑马线（横条纹）
    for i in range(6):
        rect(t, -45, 62 + i * 16, 90, 9, "white")
        rect(t, -45, -62 - i * 16 - 9, 90, 9, "white")

    t.penup()
    t.goto(-W / 2 + 12, 58)
    t.color("white")
    t.write("智慧路口平面示意图", font=("Microsoft YaHei", 14, "bold"))


def draw_lamp(t, x, y, color):
    """在 (x, y) 画一个信号灯"""
    t.clear()
    t.penup()
    t.goto(x, y)
    t.dot(30, color)


def countdown(text, screen, seconds, tip):
    """倒计时文字动画：每秒刷新一个数字，结束后显示提示语"""
    for s in range(seconds, 0, -1):       # 倒序循环：seconds → 1
        text.clear()                      # 先擦掉上一个数字，避免叠加
        text.write(str(s), align="center", font=("Arial", 30, "bold"))
        screen.update()                   # 手动刷新一帧，数字才会出现
        time.sleep(1)                     # 停 1 秒，形成"1 秒一跳"的节奏
    text.clear()
    text.write(tip, align="center", font=("Microsoft YaHei", 18, "bold"))
    screen.update()


def main():
    # 1) 屏幕对象：负责标题、窗口尺寸与背景色，不参与绘图
    screen = turtle.Screen()
    screen.title("智慧路口示意图 - turtle")
    screen.setup(W + 40, H + 40)
    screen.bgcolor("#eef1f5")
    screen.tracer(0)                      # 关闭自动刷新，手动 update

    # 2) 画笔一：只画静态场景（道路、斑马线、标题），画完不再改动
    pen = turtle.Turtle()
    pen.hideturtle()
    pen.speed(0)
    draw_scene(pen)

    # 3) 画笔二：只画信号灯圆点；切换相位时 clear() 不会擦掉场景
    lamp = turtle.Turtle()
    lamp.hideturtle()
    lamp.speed(0)

    # 4) 画笔三：只写倒计时数字与提示文字，位置固定在路口上方
    text = turtle.Turtle()
    text.hideturtle()
    text.speed(0)
    text.color("#c0392b")
    text.penup()
    text.goto(0, 150)

    # 阶段一：红灯 3 秒
    draw_lamp(lamp, 0, 105, "#c0392b")
    countdown(text, screen, 3, "红灯")

    # 阶段二：绿灯 5 秒
    draw_lamp(lamp, 0, 105, "#27ae60")
    text.color("#1e8449")
    countdown(text, screen, 5, "绿灯放行")

    screen.update()
    print("示意图绘制完成，点击窗口关闭")
    screen.exitonclick()


if __name__ == "__main__":
    main()

### 运行结果

```text
示意图绘制完成，点击窗口关闭
```

窗口中显示：灰色十字道路、棕色路口中心区、四处白色斑马线，左上角标题；路口上方一个圆点先为红色、伴随 3→2→1 倒计时，随后变绿、倒计时 5→1，最后显示"绿灯放行"。

### 代码解读

| 代码段 | 对应知识点 | 作用 |
|---|---|---|
| `def rect(t, x, y, w, h, color)` | 13.2、6.2 | 把 8 行绘制封装成一次调用，参数即"位置 + 尺寸 + 颜色" |
| `t.penup()` → `goto` → `pendown()` | 13.1 | 抬笔定位，避免拖线 |
| `for _ in range(2): forward, left, left` | 13.2 | 两次"长 + 转 90° + 宽 + 转 90°"闭合矩形 |
| `begin_fill()` / `end_fill()` | 13.2 | 路径闭合后填充 |
| 斑马线循环 `62 + i * 16` | 13.2、3.3 | 等间距排布，改 `16` 即改间距 |
| `screen.tracer(0)` | 13.3 | 关闭自动刷新，几十个矩形一次画完不闪烁 |
| 三个 `turtle.Turtle()` | 13.3、11.1 | 场景 / 灯 / 文字各一支笔，`clear()` 互不影响 |
| `t.clear()` + `dot()` | 13.2、13.3 | 每帧只重画灯，场景保留 |
| `for s in range(seconds, 0, -1)` | 13.3、3.3 | 倒序循环实现倒计时 |
| `screen.update()` + `time.sleep(1)` | 13.3 | 刷新后停 1 秒，形成动画节奏 |

**改一改**：把 `screen.tracer(0)` 注释掉再运行，观察斑马线逐条出现的过程（默认自动刷新）；把 `draw_lamp` 里的 `t.clear()` 删掉，观察信号灯是否变成"两点叠加"。

### 失败判定

| 现象 | 判定结论 | 定位方法 |
|---|---|---|
| 斑马线逐条蹦出并闪烁 | 失败：`tracer(0)` 未生效或调用过晚 | 核对 `tracer(0)` 是否早于 `draw_scene(pen)` |
| 倒计时数字叠成 `321` | 失败：每帧未先清空文字 | 查 `countdown()` 首行是否为 `text.clear()` |
| 切换到绿灯时道路消失 | 失败：三支画笔共用了一支 | 数 `turtle.Turtle()` 应为 3 次 |
| 窗口一闪就关 | 失败：缺少保持窗口的调用 | 查 `main()` 末尾是否调 `screen.exitonclick()` |
| 终端打印 `NameError`，退出码非 0 | 失败：没有执行 `main()` | 核对 `if __name__ == "__main__":` 分支 |

## 案例 13-2　pygame 路口车流仿真

- **覆盖知识点**：13.4（主循环、事件、`clock.tick`、时间驱动位移）、13.5（雏形：停止线判定与通过计数）
- **源文件**：`case2_车流仿真.py`
- **难度**：综合　**建议用时**：45 分钟

### 任务情境

值班室想直观说明红灯为何把车压到停止线前。你要在 800×400 窗口做一个最小车流仿真：4 辆车自左侧驶入、按 60 FPS 前进，相位按绿 5 秒、黄 1 秒、红 4 秒循环，红灯时车头停在 `STOP_LINE_X` 前，驶出右边界后回收计数，左上显示相位与通过车数。

### 任务要求

1. 建立 60 FPS 主循环，处理 `QUIT` 事件；
2. 用像素坐标绘制道路与停止线，用矩形表示车辆；
3. 信号灯按"绿 5 秒 — 黄 1 秒 — 红 4 秒"循环切换；
4. 车辆位置与速度用 `x += speed` 更新，`x` 超过屏幕则回收并计数；
5. 在窗口左上角显示统计信息。

> ⚠️ **运行提示**：本段依赖 `pygame`，运行前先执行 `%pip install pygame`。仿真窗口需手动关闭（点右上角 ×）。

### 输入规格

| 项 | 要求 |
|---|---|
| 数据来源 | 无外部输入；窗口尺寸、帧率、道路边界、停止线、车辆初值与相位时长均为字面量 |
| 类型 | `int` 与 `float` 字面量；`cars` 元素为 `[x, 车道y, 速度]` 三元素列表，按下标原地修改 |
| 取值范围 | `W` 为 800、`H` 为 400；`FPS` 为 60；道路纵向 150—250；`STOP_LINE_X` 为 620；4 辆车初始 x 为 0、120、240、360，车道 y 为 175，速度 3.0；绿 5 秒、黄 1 秒、红 4 秒 |
| 非法值 | `FPS` 改成 0、速度改成 0、`RED_T` 写成负数、`cars` 写成空列表、用 `input()` 读相位时长 |

### 输出规格

| 项 | 要求 |
|---|---|
| 行数 | 终端无程序自身输出：源码不含 `print`，退出时执行 `sys.exit()`；pygame 版本提示不计 |
| 窗口尺寸 | `set_mode((800, 400))`，标题 `智慧路口车流仿真`；每帧先 `fill("#eef1f5")` 清屏 |
| 绘制元素 | 灰色路面 800×100（y 150—250）、y=200 白线宽 2、停止线 x=620 宽 4、(300—340, 150) 短白线宽 6、4 辆 40×30 蓝车、(60, 70) 半径 26 圆点、左上 (100, 50) 统计文字 |
| 刷新次数 | 每帧末尾 1 次 `flip()`；60 FPS 下每秒 60 次，10 秒约 600 次 |
| 精度 / 单位 | 位置单位为像素；`dt` 单位为秒，位移按 `speed * 60 * dt` 累加；绿黄红合计 10 秒一轮 |
| 关闭方式 | 点击关闭按钮触发 `QUIT`，循环后执行 `pygame.quit()` 与 `sys.exit()` |
| 判定方式 | 终端无 `Traceback`；截图对照元素与坐标；圆点按绿→黄→红约 10 秒一轮；关闭按钮正常退出 |

### 参考代码

In [ ]:
# ---------- 第 13 章 综合项目实战 · 案例 13-2：pygame 路口车流仿真 ----------
# 对应知识点：13.4 pygame 主循环 / 事件 / clock.tick 帧率 / 时间驱动位移、13.5 停止线判定与通过计数
# 功能说明：车辆按帧率匀速通过路口，红灯时在停止线前等待，信号灯按相位循环并统计通过车数

import pygame
import sys

W, H = 800, 400
FPS = 60
# 道路上下边界：pygame 的 y 轴向下为正，数值越大越靠下
ROAD_TOP, ROAD_BOTTOM = 150, 250
STOP_LINE_X = 620                        # 停止线横坐标（红灯时车头不得越过）

pygame.init()
screen = pygame.display.set_mode((W, H))
pygame.display.set_caption("智慧路口车流仿真")
clock = pygame.time.Clock()              # 帧率控制器：tick() 返回距上一帧的毫秒数
font = pygame.font.SysFont("microsoftyahei", 22)

# 信号灯状态：green / yellow / red
light_state = "green"
light_timer = 0                          # 当前相位已持续的秒数
GREEN_T, YELLOW_T, RED_T = 5, 1, 4

# 车辆列表：每个元素为 [x, 车道y, 速度]
cars = [[i * 120.0, ROAD_TOP + 25, 3.0] for i in range(4)]
passed = 0


def current_phase():
    """返回当前相位应持续的秒数，供计时判断使用"""
    if light_state == "green":
        return GREEN_T
    elif light_state == "yellow":
        return YELLOW_T
    return RED_T


running = True
while running:                           # 主循环：每一轮就是动画的一帧
    dt = clock.tick(FPS) / 1000.0          # 本帧耗时（秒），把帧率差异折算进位移

    # 事件处理：关闭窗口即退出循环，避免程序无法结束
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # ---------- 信号灯计时 ----------
    light_timer += dt
    if light_timer >= current_phase():     # 本相位到时，按 绿→黄→红→绿 轮转
        light_timer = 0
        if light_state == "green":
            light_state = "yellow"
        elif light_state == "yellow":
            light_state = "red"
        else:
            light_state = "green"

    # ---------- 车辆运动 ----------
    for car in cars:
        x, y, speed = car
        front = x + 40
        if light_state == "red" and front < STOP_LINE_X and front + speed * 60 >= STOP_LINE_X - 5:
            pass                            # 接近停止线且红灯：本帧不前进
        else:
            car[0] += speed * 60 * dt

        if car[0] > W:
            car[0] = -60
            passed += 1

    # ---------- 绘制 ----------
    screen.fill("#eef1f5")
    pygame.draw.rect(screen, "#5b5b5b", (0, ROAD_TOP, W, ROAD_BOTTOM - ROAD_TOP))
    pygame.draw.line(screen, "white", (0, 200), (W, 200), 2)
    pygame.draw.line(screen, "white", (STOP_LINE_X, ROAD_TOP), (STOP_LINE_X, ROAD_BOTTOM), 4)
    pygame.draw.line(screen, "white", (300, ROAD_TOP), (340, ROAD_TOP), 6)

    for car in cars:
        pygame.draw.rect(screen, "#2980b9", (int(car[0]), car[1] - 15, 40, 30), border_radius=6)

    color = {"green": "#27ae60", "yellow": "#f1c40f", "red": "#c0392b"}[light_state]
    pygame.draw.circle(screen, color, (60, 70), 26)

    info = "相位：" + light_state + "   已通过：" + str(passed) + " 辆"
    screen.blit(font.render(info, True, "#2c3e50"), (100, 50))

    pygame.display.flip()

pygame.quit()
sys.exit()

### 运行结果

```text
（弹出 800×400 窗口）
左上角圆点按 绿5s → 黄1s → 红4s 循环变色
4 辆蓝色车辆自左向右行驶，红灯时在停止线前停住
右上角文字实时显示：相位：green   已通过：12 辆
关闭窗口后程序退出
```

### 代码解读

| 代码段 | 对应知识点 | 作用 |
|---|---|---|
| `pygame.init()` + `set_mode` | 13.4 | 初始化并建 800×400 画布 |
| `clock.tick(FPS) / 1000.0` | 13.4 | 得到秒级 `dt`，位移与帧率解耦 |
| `for event in pygame.event.get()` | 13.4 | 处理关闭事件，否则窗口无响应 |
| `light_timer += dt` + 三路切换 | 13.4、3.2 | 用累计时间控制相位，等效有限状态机 |
| `current_phase()` | 13.4、6.1 | 把"当前相位时长"封装成函数，主循环更干净 |
| `car[0] += speed * 60 * dt` | 13.4 | 时间驱动位移：`speed` 单位为"像素/帧"，乘 60 折算为像素/秒 |
| 停止线判断 `front + speed * 60 >= STOP_LINE_X - 5` | 13.5 | 预判"下一帧是否会越过停止线"，红灯时不动 |
| `if car[0] > W: car[0] = -60; passed += 1` | 13.5 | 循环复用车辆 + 通过计数 |
| `screen.fill()` 每帧清屏 | 13.4 | 消除拖影 |
| `pygame.display.flip()` | 13.4 | 提交本帧画面 |
| `{"green": …, "red": …}[light_state]` | 13.4、5.5 | 字典映射颜色，替代 `if` 链 |

**改一改**：把 `dt` 改为固定 `0.016`（去掉 `clock.tick` 的返回值），观察不同性能机器上的速度差异；把 `RED_T` 改为 10，观察停止线前的排队长度变化。

### 失败判定

| 现象 | 判定结论 | 定位方法 |
|---|---|---|
| 窗口无响应 | 失败：主循环没有取事件 | 查 `event.get()` 与 `QUIT` 分支是否在循环内 |
| 车辆拖出蓝色残影 | 失败：每帧没有清屏 | 查 `screen.fill("#eef1f5")` 是否在绘制之前 |
| 窗口全黑 | 失败：缺少提交画面的调用 | 查循环末尾是否有 `pygame.display.flip()` |
| 相位切换明显快于 5 / 1 / 4 秒 | 失败：用帧计数代替累计秒数 | 查 `light_timer += dt` 与 `current_phase()` |
| 红灯时车辆仍驶过停止线 | 失败：停止线预判条件写错 | 核对 `front + speed * 60 >= STOP_LINE_X - 5` |
| 每辆车都没被计成过车记录 | 失败：计数写在回收分支之外 | 查 `if car[0] > W:` 内是否同时写 `passed += 1` |
| 报错 `ModuleNotFoundError: No module named 'pygame'` | 失败：依赖未安装 | 执行 `pip install pygame` 后重跑 |

---

# 第三部分　布置练习

> **环境提醒**：两个练习都会打开独立窗口，**不要**放在自动批改脚本中运行；提交时附上运行截图与源码。

## 练习 13-1　turtle 绘制智慧停车场

- **对应知识点**：13.1（坐标与画笔）、13.2（矩形与文字）、13.3（`tracer` / `update`）
- **提交文件**：`practice1_停车场平面图.py`
- **难度**：综合　**建议用时**：35 分钟

### 任务情境

园区停车场要换一块引导屏：3 行 6 列、每格 60×100、间距 8 像素，当晚有 5 个车位已占用，需标出车牌尾号，其余 13 格写 `FREE`。你要用 turtle 出一张居中的平面图，标题写明「已用 5 / 18」，且绘制过程不能闪烁。

### 题目

绘制一个 3×6 的停车场平面图：每格为一个车位，已占用车位用蓝色填充并标注车牌尾号，空车位用白色填充并写 `FREE`。

### 要求

1. 车位尺寸 60×100（宽×高），间距 8 像素，整体居中；
2. 用嵌套循环绘制车位矩阵；
3. 用列表保存已占用车位的坐标（行列），在对应车位中写字；
4. 标题显示"智慧停车场 · 已用 5 / 18"。

### 提示

- 先算出停车场总宽高，再反推左下角起点，才能保证整体居中：`start_x = -total_w / 2`。
- 行号自上而下时，`y` 坐标要反着算：`y = start_y + (ROWS - 1 - r) * (SLOT_H + GAP)`。
- 判断"某格是否已占用"用 `(r, c) in occupied`，`occupied` 是元组列表。

### 自检要点（不含答案）

- [ ] 3 行 6 列共 18 个车位，整体居中不偏；
- [ ] 5 个占用车位为浅蓝底 + 车牌尾号，其余为白底 + `FREE`；
- [ ] 标题居中于顶部，数字为 5 / 18；
- [ ] 使用 `tracer(0)` + `update()`，绘制过程无闪烁。

### 测试用例表

| 编号 | 类别 | 输入 | 关键预期输出 | 判定方式 |
|---|---|---|---|---|
| T1 | 正常 | 不改参数直接运行 `practice1_停车场平面图.py` | 窗口 760×560；18 个车位，5 个浅蓝带尾号、13 个白底 `FREE`；标题 5 / 18 | 与答案 13-1「运行结果」截图对照 |
| T2 | 边界 | 把 `occupied` 改成空列表 | 18 个车位全为白底 `FREE`，标题 `已用 0 / 18` | 浅蓝格为 0，标题为 0 / 18 |
| T3 | 异常 | 把 `plates` 改成只有 3 个元素 | 不报 `IndexError`，5 个车位复用 3 个尾号 | 退出码 0，尾号仅 3 种 |

### 验收标准表

| 检查项 | 判定方法 | 通过标准 | 分值 |
|---|---|---|---|
| 矩阵与居中 | 截图数格子并比左右留白 | 18 格，整体居中 | 4 |
| 占用与标注 | 数浅蓝格与白格 | 5 个尾号、13 个 `FREE`，与 `occupied` 对应 | 3 |
| 封装与刷新 | 阅读源码并观察绘制 | 有 `rect()`、`tracer(0)` 与单次 `update()` | 2 |
| 标题 | 截图看顶部文字 | 居中显示 `已用 5 / 18` | 1 |
| 合计 | — | — | 10 |

## 练习 13-2　给车流仿真加上碰撞检测与计分

- **对应知识点**：13.5（`Rect.colliderect`、分组、计分）、13.4（主循环改造）、第 7 章（`sorted(key=)`）
- **提交文件**：`practice2_碰撞计分.py`
- **难度**：进阶　**建议用时**：45 分钟

### 任务情境

领导要求把「追尾」纳入考核：同车道前后车矩形重叠扣 5 分，每通过一辆车加 1 分，得分与碰撞次数实时显示在 800×400 窗口。你要先按 `y` 分组、按 `x` 排序，再用 `colliderect()` 判重叠，并让后车退到前车之后，避免重复计碰撞。

### 题目

在案例 13-2 的基础上增加两件事：① 同一车道前后车距离过近（小于车长）时判定为追尾，分数扣 5 并提示；② 每通过一辆车加 1 分，分数与碰撞次数显示在界面上。

### 要求

1. 车辆改为每车道一条队列，按 `y` 分组；
2. 用 `pygame.Rect` 的 `colliderect()` 判断两车是否重叠；
3. 发生碰撞后把后车速度降为 0，并计入碰撞次数；
4. 界面显示"得分 / 碰撞次数"。

### 提示

- 先把车辆按车道分组（字典 `{ y: [车辆…] }`），只比较**同一车道**的相邻两车，避免把对向车道的车误判为追尾。
- 分组后要按 `x` **排序**，否则"相邻"不是真正的前后车。
- 碰撞后若只把速度降为 0 而不回退位置，两车仍重叠，下一帧会再次计碰撞——必须把后车 `x` 退到前车之后。

### 自检要点（不含答案）

- [ ] 碰撞只发生在同车道相邻车之间；
- [ ] 碰撞后后车停住并回退，不会连续多帧重复计碰撞；
- [ ] 通过一辆车得分 +1，碰撞一次得分 −5；
- [ ] 界面实时显示得分与碰撞次数。

### 测试用例表

| 编号 | 类别 | 输入 | 关键预期输出 | 判定方式 |
|---|---|---|---|---|
| T1 | 正常 | 按题目加分组、排序、碰撞与计分后直接运行 | 窗口 800×400；两车道各 3 辆；左上文字刷新；追尾后后车停住并回退 | 截图对照，终端无报错 |
| T2 | 边界 | 每车道车辆数由 `range(3)` 改成 `range(1)` | 每车道 1 辆，`range(len(same) - 1)` 为空，碰撞数恒为 0 | 运行 3 秒，碰撞为 0、得分大于 0 |
| T3 | 异常 | 删掉某一辆车的 `"v"` 键 | 运行即报错，退出码非 0，含 `KeyError: 'v'` | 报错栈指向 `car["v"] * 60 * dt` |

### 验收标准表

| 检查项 | 判定方法 | 通过标准 | 分值 |
|---|---|---|---|
| 分组与排序 | 阅读源码 | 按 `LANES` 分组后调用 `sort(key=...)` | 4 |
| 碰撞处理 | 制造一次追尾后连读 3 秒 | 后车停住并回退 42 像素，碰撞只加 1 | 3 |
| 计分 | 观察通过一辆与追尾一次 | 过车 +1，追尾 −5 | 2 |
| 界面显示 | 截图左上角文字 | 得分与碰撞次数实时刷新 | 1 |
| 合计 | — | — | 10 |

---

# 第四部分　习题答案

## 答案 13-1　turtle 绘制智慧停车场

> 对应 **练习 13-1**（第三部分）

### 参考代码

In [ ]:
# ---------- 第 13 章 综合项目实战 · 练习 13-1 参考答案：turtle 绘制智慧停车场 ----------
# 对应知识点：13.1 坐标与画笔、13.2 矩形与文字、13.3 tracer(0) + update()
# 功能说明：绘制 3×6 车位矩阵并整体居中，已占用车位标注车牌尾号、空位标注 FREE

import turtle

ROWS, COLS = 3, 6                        # 车位矩阵：3 行 6 列，共 18 个车位
SLOT_W, SLOT_H, GAP = 60, 100, 8         # 单车位宽、高与车位间距（像素）

# 已占用车位用（行, 列）表示，下标从 0 开始；plates 与之一一对应
occupied = [(0, 0), (0, 2), (1, 1), (1, 4), (2, 5)]
plates = ["2345", "7890", "4680", "1122", "5667"]


def rect(t, x, y, w, h, color, text="", text_color="#2c3e50"):
    """以 (x, y) 为左下角画填充矩形，可选在中心写一行文字"""
    t.penup()
    t.goto(x, y)
    t.pendown()
    t.color("#34495e")
    t.fillcolor(color)
    t.begin_fill()
    for _ in range(2):
        t.forward(w)
        t.left(90)
        t.forward(h)
        t.left(90)
    t.end_fill()
    if text != "":
        t.penup()
        t.goto(x + w / 2, y + h / 2 - 8)
        t.color(text_color)
        t.write(text, align="center", font=("Arial", 12, "bold"))


def main():
    screen = turtle.Screen()
    screen.title("智慧停车场平面图")
    screen.setup(760, 560)
    screen.bgcolor("#f4f6f8")
    screen.tracer(0)

    t = turtle.Turtle()
    t.hideturtle()
    t.speed(0)

    # 先算出停车场总宽高，再由左下角起点反推每个车位的坐标，整体才能居中
    total_w = COLS * SLOT_W + (COLS - 1) * GAP
    total_h = ROWS * SLOT_H + (ROWS - 1) * GAP
    start_x = -total_w / 2
    start_y = -total_h / 2

    idx = 0
    for r in range(ROWS):
        for c in range(COLS):
            x = start_x + c * (SLOT_W + GAP)
            # 行号自上而下，而 turtle 的 y 向上为正，故用 (ROWS - 1 - r) 反向取行
            y = start_y + (ROWS - 1 - r) * (SLOT_H + GAP)
            if (r, c) in occupied:
                label = "京A" + plates[idx % len(plates)][-4:]
                idx += 1
                rect(t, x, y, SLOT_W, SLOT_H, "#aed6f1", label, "#1a5276")
            else:
                rect(t, x, y, SLOT_W, SLOT_H, "white", "FREE", "#7f8c8d")

    t.penup()
    t.goto(0, total_h / 2 + 40)
    t.color("#2c3e50")
    t.write("智慧停车场 · 已用 " + str(len(occupied)) + " / " + str(ROWS * COLS),
            align="center", font=("Microsoft YaHei", 18, "bold"))

    screen.update()
    screen.exitonclick()


if __name__ == "__main__":
    main()

### 运行结果

```text
（弹出 760×560 窗口）
3 行 6 列车位，5 个浅蓝车位分别标注 京A2345 等，其余 13 个白底写 FREE
顶部标题：智慧停车场 · 已用 5 / 18
```

### 解析

1. **居中的算法**：先算总宽 `6×60 + 5×8 = 400`、总高 `3×100 + 2×8 = 316`，起点取 `-总宽/2`，于是图形以原点为中心。这是"先算尺寸、再定起点"的通用套路。
2. **行号与 y 的反向关系**：第 0 行在最上方，其 y 应为最大值，故 `(ROWS - 1 - r)`。若直接用 `r`，图形会上下颠倒。
3. **`rect()` 增加文字参数**：把"画框 + 写字"合并，避免在嵌套循环里重复写两套定位代码；`text=""` 的默认值让"只画框"的调用仍然简洁。
4. **`idx % len(plates)`**：占用车位 5 个、车牌尾号 5 个，取模是为了防止占用数超过列表长度时报 `IndexError`。
5. **`tracer(0)` + 单次 `update()`**：18 个矩形 + 18 段文字，若逐笔刷新会明显卡顿；关闭自动刷新后一次性呈现。

### 常见错误

| 错误现象 | 原因 | 修正 |
|---|---|---|
| 图形偏右下 | 起点用了 0 而非 `-总宽/2` | 起点居中 |
| 行序颠倒 | `y` 用 `r` 直接算 | `(ROWS - 1 - r)` |
| 文字压在边框上 | 写字前未 `penup()` | 写字前抬笔 |
| 中文显示方块 | 字体名不存在 | 用"Microsoft YaHei"或"SimHei" |

### 评分要点（供批改）

| 项 | 分值 | 观察点 |
|---|---|---|
| 矩阵与居中 | 4 | 18 格、整体居中 |
| 占用与标注 | 3 | 5 个蓝底车牌、13 个 FREE |
| 封装与刷新 | 2 | `rect()` 复用、`tracer`/`update` |
| 标题 | 1 | 5 / 18 居中显示 |

### 与测试用例的对应

| 用例 | 本答案的对应结果 | 说明 |
|---|---|---|
| T1 | 18 个车位、5 个浅蓝尾号、13 个 `FREE`，标题 5 / 18 | 与「运行结果」一致 |
| T2 | `occupied` 置空后仍画 18 个车位，全为白底，标题变 `已用 0 / 18` | 标题用 `len(occupied)` 拼接 |
| T3 | `plates` 只有 3 个元素时用 `idx % len(plates)` 循环取用 | 不报 `IndexError` |

## 答案 13-2　给车流仿真加上碰撞检测与计分

> 对应 **练习 13-2**（第三部分）

### 参考代码

In [ ]:
# ---------- 第 13 章 综合项目实战 · 练习 13-2 参考答案：给车流仿真加碰撞检测与计分 ----------
# 对应知识点：13.5 pygame.Rect 与 colliderect() 碰撞检测、13.4 主循环改造、7.3 sorted(key=)
# 功能说明：同车道前后车矩形重叠即判定追尾并扣分，车辆通过路口加分，界面实时显示得分

import pygame
import sys

W, H = 800, 400
FPS = 60
LANES = [175, 225]                       # 两条车道的中心 y 坐标

pygame.init()
screen = pygame.display.set_mode((W, H))
pygame.display.set_caption("智慧路口车流仿真 · 碰撞检测")
clock = pygame.time.Clock()
font = pygame.font.SysFont("microsoftyahei", 20)

cars = []
for i, y in enumerate(LANES):
    for k in range(3):
        # 每条车道 3 辆车，初始 x 互相错开 150 像素，避免开局就重叠
        cars.append({"x": -k * 150.0 - i * 40, "y": y, "v": 3.0})

score = 0
crashes = 0


def rect_of(car):
    """把车辆字典转成 pygame.Rect，用于碰撞检测（x, y, 宽, 高）"""
    return pygame.Rect(int(car["x"]), car["y"] - 15, 40, 30)


running = True
while running:
    dt = clock.tick(FPS) / 1000.0
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    for car in cars:
        car["x"] += car["v"] * 60 * dt
        if car["x"] > W:
            car["x"] = -60
            score += 1

    # 同车道碰撞检测：先按车道分组，再按 x 排序，只有相邻两车才可能追尾
    for y in LANES:
        same = [c for c in cars if c["y"] == y]
        same.sort(key=lambda c: c["x"])
        for i in range(len(same) - 1):
            if rect_of(same[i]).colliderect(rect_of(same[i + 1])):
                # 矩形重叠 → 判定追尾：后车停住、扣分、并退到前车之后
                same[i + 1]["v"] = 0
                crashes += 1
                score -= 5
                same[i + 1]["x"] = same[i]["x"] - 42   # 回退，避免下一帧重复计数
            else:
                same[i + 1]["v"] = 3.0                 # 未碰撞则恢复正常速度

    screen.fill("#eef1f5")
    pygame.draw.rect(screen, "#5b5b5b", (0, 150, W, 100))
    for y in LANES:
        pygame.draw.line(screen, "white", (0, y + 25), (W, y + 25), 1)

    for car in cars:
        pygame.draw.rect(screen, "#2980b9", rect_of(car), border_radius=6)

    screen.blit(font.render("得分：" + str(score) + "   碰撞：" + str(crashes), True, "#2c3e50"), (20, 20))
    pygame.display.flip()

pygame.quit()
sys.exit()

### 运行结果

```text
（弹出 800×400 窗口）
两条车道各 3 辆车自左向右行驶
发生追尾时后车停住并回退，碰撞数 +1
车辆驶出屏幕后重新进入，得分 +1
左上角实时显示：得分：14   碰撞：2
```

### 解析

1. **分组再排序**：`[c for c in cars if c["y"] == y]` 取出同车道车辆，`sort(key=lambda c: c["x"])` 让"相邻"等于"前后"。少了排序，比较的是列表顺序而非位置顺序，会漏判或误判。
2. **为什么必须回退位置**：只把 `v` 设为 0，车辆仍与前车重叠，下一帧 `colliderect` 依然为真，碰撞数会每帧 +1（一秒钟 60 次）。回退 42 像素（车长 40 + 间隙 2）后脱离重叠。
3. **恢复速度**：`else:` 分支把 `v` 恢复为 3.0，车辆在前车驶离后能重新启动，形成"走走停停"的仿真效果。
4. **计分口径**：通过一辆 +1（`score += 1`），追尾一次 −5（`score -= 5`），两者在界面上实时显示。若只加不减，得分无法反映仿真质量，这是本题"计分"要求的关键。
5. **`rect_of()` 封装**：把"字典 → 矩形"的转换写成一个函数，绘制与碰撞判定共用同一份几何定义，避免两处尺寸不一致。

### 常见错误

| 错误现象 | 原因 | 修正 |
|---|---|---|
| 碰撞数每秒涨几十 | 未回退位置 | 后车 `x` 退到前车之后 |
| 对向车道也计碰撞 | 未按 `y` 分组 | 分组后再比较 |
| 碰撞后车永久停住 | 缺少恢复速度的 `else` | 脱离重叠后恢复 `v` |
| 扣分未生效 | 只在注释里写了"扣 5 分" | 碰撞分支加 `score -= 5` |

### 评分要点（供批改）

| 项 | 分值 | 观察点 |
|---|---|---|
| 分组与排序 | 4 | 同车道、按 `x` 有序 |
| 碰撞处理 | 3 | 减速 + 回退，不重复计 |
| 计分 | 2 | 通过 +1、碰撞 −5 |
| 界面显示 | 1 | 得分与碰撞数实时刷新 |

### 与测试用例的对应

| 用例 | 本答案的对应结果 | 说明 |
|---|---|---|
| T1 | 两车道各 3 辆，追尾时后车速度置 0 并回退 42 像素，左上显示得分与碰撞 | 正常路径 |
| T2 | 每车道 1 辆时 `range(len(same) - 1)` 为空，碰撞数恒为 0 | 分组长度为 1 |
| T3 | 缺 `"v"` 键在 `car["v"] * 60 * dt` 处抛 `KeyError` | 属预期异常 |

---

## 本章小结

- **两套坐标系**：turtle 原点在中心、y 向上；pygame 原点在左上、y 向下。换库时先换算坐标。
- **turtle 三件套**：`penup/goto/pendown` 定位、`begin_fill/end_fill` 填充、`write` 写字；重复图形必须封装成函数。
- **动画**：`tracer(0)` 关自动刷新 + 每帧 `update()`；分画笔 `clear()` 才能局部更新。
- **pygame 骨架**：初始化 → 建窗口 → 主循环（事件 → 更新 → 绘制 → flip）；位移用 `dt` 而非帧数。
- **碰撞**：`Rect.colliderect` 判重叠；先按车道分组、再按 `x` 排序；碰撞后必须"减速 + 回退"。

## 自检清单

- [ ] 能说出 turtle 与 pygame 坐标系的两点差异；
- [ ] 能封装一个 `rect()` 函数并解释 6 个参数；
- [ ] 会写 `tracer(0)` + `update()` 的动画循环，并说明为何要分画笔；
- [ ] 能默写 pygame 六步骨架与 `clock.tick` 的作用；
- [ ] 会用 `colliderect` 做碰撞判定，并说明"分组 + 排序 + 回退"三个必要步骤。

## 前后章节衔接

- 本章把第 3 章（循环）、第 6 章（函数封装）、第 11—12 章（对象与继承）综合为"可视作品"。
- 本章的分画笔、`rect_of()` 封装，是"单一职责"在图形编程中的体现，与第 6 章练习 6-2 的重构标准一致。
- 学完本章后进入**终结性考核**：回归文本数据处理，用第 1—12 章知识完成"读入 → 清洗 → 统计 → 决策 → 报表"闭环。

- **补充（2026-11-14 知识点体系优化）**：本轮在第 13 章之后新增**第 14 章「本地 AI 应用前置技能」**与**第 15 章「本地 AI 综合实战」**，构成 22 学时的本地 AI 拓展模块（不占课标模块学时、不纳入终结性考核范围，可按专业方向与机房显卡条件选做）。上面第三条是 v2.1 结构下的表述；现行教学顺序为 **13 → 14 → 15 → 99**，终结性考核仍排在最后，考核范围与命题原则均不变。第 14 章会把本章用到的 `pygame` 依赖管理补上系统讲解（`venv` / `pip` / `requirements.txt`），并补齐 `pathlib`、`json`、`urllib.request` 等工程技能。

---

[🏠 课程首页](../00_课程索引.ipynb)&nbsp;&nbsp;|&nbsp;&nbsp;[◀ 上一章：面向对象编程下](../12面向对象编程下/12_面向对象编程下.ipynb)&nbsp;&nbsp;|&nbsp;&nbsp;[下一章：本地AI应用前置技能 ▶](../14本地AI应用前置技能/14_本地AI应用前置技能.ipynb)